# Analytics Agent Backend - Colab Setup

FastAPI backend ko Google Colab pe run karne ke liye.

**Features:**
- Public URL via Cloudflare Tunnel
- 12-16 GB RAM (Colab free)
- Auto-reconnect keep-alive
- Test API endpoint instantly

## Step 1: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn[standard] pydantic pydantic-settings httpx PyJWT \
    pandas numpy scipy statsmodels scikit-learn matplotlib openpyxl xlrd \
    python-multipart cryptography psycopg[binary] pyarrow openai nest-asyncio pyngrok

## Step 2: Upload Backend Code

Yeh cell run karne se file upload dialog khulega. Upload karo:
- Poora `app/` folder (zip karke)

In [ ]:
from google.colab import files
import zipfile
import os

print("📤 Upload app.zip (backend/app folder ko zip karke)")
uploaded = files.upload()

# Extract
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✅ Extracted {filename}")

!ls -la

## Step 3: Configure Environment Variables

In [ ]:
import os
from cryptography.fernet import Fernet

# === CONFIGURE THESE ===
os.environ['SUPABASE_URL'] = 'https://your-project.supabase.co'
os.environ['SUPABASE_SERVICE_ROLE_KEY'] = 'your-service-role-key'
os.environ['SUPABASE_PUBLISHABLE_KEY'] = 'your-anon-key'
os.environ['SUPABASE_JWT_SECRET'] = 'your-jwt-secret'

# Auto-generate SQL secrets key
os.environ['SQL_SECRETS_KEY'] = Fernet.generate_key().decode()

# Optional: OpenAI for LLM features
os.environ['LLM_ENABLED'] = 'false'
os.environ['OPENAI_API_KEY'] = ''

# App config
os.environ['APP_ENV'] = 'development'
os.environ['CORS_ORIGINS'] = '*'
os.environ['MAX_UPLOAD_MB'] = '100'
os.environ['ALLOWED_FILE_EXTENSIONS'] = 'csv,xls,xlsx'
os.environ['SQL_ALLOWED_HOSTS'] = '*'
os.environ['JOB_CONCURRENCY'] = '2'
os.environ['JOB_POLL_INTERVAL_SECONDS'] = '1.0'

print("✅ Environment configured")
print(f"🔑 Generated SQL_SECRETS_KEY: {os.environ['SQL_SECRETS_KEY'][:20]}...")

## Step 4: Setup Cloudflare Tunnel (Public URL)

In [ ]:
# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!mv cloudflared-linux-amd64 /usr/local/bin/cloudflared

print("✅ Cloudflared installed")

## Step 5: Start Backend Server

In [ ]:
import nest_asyncio
import threading
import subprocess
import time
import re
from IPython.display import clear_output, HTML

nest_asyncio.apply()

# Start uvicorn in background
def start_uvicorn():
    import uvicorn
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000, log_level="info")

uvicorn_thread = threading.Thread(target=start_uvicorn, daemon=True)
uvicorn_thread.start()

print("⏳ Starting FastAPI server...")
time.sleep(5)
print("✅ FastAPI running on http://localhost:8000")

# Start cloudflared tunnel
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

# Extract public URL
public_url = None
for line in tunnel_process.stdout:
    print(line.strip())
    if 'trycloudflare.com' in line or 'https://' in line:
        match = re.search(r'https://[^\s]+', line)
        if match:
            public_url = match.group(0)
            break

if public_url:
    clear_output(wait=True)
    display(HTML(f'''
    <div style="padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; margin: 10px 0;">
        <h2 style="color: #2e7d32; margin: 0 0 10px 0;">🚀 Backend is LIVE!</h2>
        <p style="margin: 5px 0;"><strong>Public URL:</strong></p>
        <p style="margin: 5px 0; font-family: monospace; background: white; padding: 10px; border-radius: 4px;">
            <a href="{public_url}" target="_blank">{public_url}</a>
        </p>
        <p style="margin: 5px 0;"><strong>API Docs:</strong></p>
        <p style="margin: 5px 0; font-family: monospace; background: white; padding: 10px; border-radius: 4px;">
            <a href="{public_url}/docs" target="_blank">{public_url}/docs</a>
        </p>
        <p style="margin: 15px 0 5px 0; color: #666; font-size: 14px;">
            ⚠️ Is URL ko Android app ke <code>ANALYTICS_AGENT_API_URL</code> me daalo
        </p>
    </div>
    '''))
    
    print(f"\n✅ Backend URL: {public_url}")
    print(f"📖 API Docs: {public_url}/docs")
    print(f"\n💡 Android config me daalo:")
    print(f"   ANALYTICS_AGENT_API_URL={public_url}")
else:
    print("⚠️ Could not extract public URL. Check tunnel logs above.")

## Step 6: Keep-Alive (Tab ko khula rakhein)

In [ ]:
%%javascript
// Keep Colab session alive
function KeepAlive() {
    console.log("⏰ Keep-alive ping");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepAlive, 60000); // Every 60 seconds
console.log("✅ Keep-alive started");

## Test API

In [ ]:
import requests

# Test health endpoint
response = requests.get('http://localhost:8000/health')
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")

---

## 📝 Notes:

1. **Public URL** upar display hoga - isko Android app me use karo
2. **Tab khula rakhna** jaruri hai (keep-alive JS chalti hai)
3. **~12 hours** ke baad Colab restart hoga (free tier limit)
4. **Files upload** karne ke liye `/docs` endpoint use karo
5. **Supabase credentials** Step 3 me daalo (compulsory)

## 🔄 Agar session disconnect ho:
- `Runtime → Run all` karo
- Naya public URL milega (Android config update karna padega)

## 🚀 Production ke liye:
- Railway/Render/Google Cloud Run use karo
- Permanent URL milega